# Transformation - Combining flight data and weather data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import os
import logging
import pandas as pd

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (SparkSession
    .builder
    .appName("CombiningTransformation")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate())

spark.sparkContext.setLogLevel("FATAL")
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/rwjlb/.ivy2.5.2/cache
The jars for the packages stored in: /home/rwjlb/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c8d12ef3-e7fa-4930-b63d-9f590dc87291;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
:: resolution report :: resolve 487ms :: artifacts dl 22ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-a

In [4]:
flight_df = spark.read.parquet("s3a://rwc-ml-datasets/cleaned/Flight_Delay_Prediction_Datasets/flight_data/")
weather_df = spark.read.parquet("s3a://rwc-ml-datasets/staging/Flight_Delay_Prediction_Datasets/weather_data/weather_data.parquet")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

In [5]:
flight_df.limit(5).toPandas()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER,ORIGIN_AIRPORT_ID,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_FIPS,ORIGIN_STATE_NM,DEST_AIRPORT_ID,DEST,DEST_CITY_NAME,DEST_STATE_ABR,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,DEP_DELAY_GROUP,DEP_TIME_BLK,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,ARR_DELAY_GROUP,ARR_TIME_BLK,CRS_ELAPSED_TIME,AIR_TIME,DISTANCE,DISTANCE_GROUP,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,12:05,11:59,-6.0,0.0,0.0,-1,1200-1259,18.0,12:17,14:36,9.0,15:11,14:45,-26.0,0.0,0.0,-2,1500-1559,366.0,319.0,2611.0,11,0.0,0.0,0.0,0.0,0.0
1,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,15:20,15:38,18.0,18.0,1.0,1,1500-1559,11.0,15:49,23:44,9.0,23:54,23:53,-1.0,0.0,0.0,-1,2300-2359,334.0,295.0,2475.0,10,0.0,0.0,0.0,0.0,0.0
2,2025,3,9,1,1,2025-09-01,AA,AA,10721,BOS,"Boston, MA",MA,25,Massachusetts,12892,LAX,"Los Angeles, CA",CA,18:07,19:01,54.0,54.0,1.0,3,1800-1859,17.0,19:18,21:34,13.0,21:21,21:47,26.0,26.0,1.0,1,2100-2159,374.0,316.0,2611.0,11,6.0,0.0,0.0,0.0,20.0
3,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,10721,BOS,"Boston, MA",MA,08:35,09:09,34.0,34.0,1.0,2,0800-0859,17.0,09:26,17:46,12.0,17:17,17:58,41.0,41.0,1.0,2,1700-1759,342.0,320.0,2611.0,11,34.0,0.0,7.0,0.0,0.0
4,2025,3,9,1,1,2025-09-01,AA,AA,12892,LAX,"Los Angeles, CA",CA,6,California,12478,JFK,"New York, NY",NY,22:52,22:47,-5.0,0.0,0.0,-1,2200-2259,19.0,23:06,06:57,15.0,07:29,07:12,-17.0,0.0,0.0,-2,0700-0759,337.0,291.0,2475.0,10,0.0,0.0,0.0,0.0,0.0


In [6]:
weather_df.limit(5).toPandas()

,station,timestamp,air_temp_f,dew_point_temp_f,relative_humidity,wind_speed_kts,wind_gust_kts,visibility_miles,sky_l1_coverage,sky_l2_coverage,sky_l3_coverage,present_weather_codes,1hr_precipitation_inches,pressure_altimeter_inches
0,HTS,2025-01-01 00:51:00,46.00,37.00,70.64,15.00,25.00,10.00,FEW,BKN,OVC,M,0.01,29.66
1,SAN,2025-01-01 00:51:00,57.00,49.00,74.55,4.00,M,9.00,BKN,M,M,M,0.00,30.05
2,SGF,2025-01-01 00:52:00,36.00,29.00,75.46,11.00,M,10.00,OVC,M,M,M,0.00,30.14
3,BTR,2025-01-01 00:53:00,62.00,39.00,42.56,4.00,M,10.00,CLR,M,M,M,0.00,30.03
4,HOU,2025-01-01 00:53:00,64.00,40.00,41.25,4.00,M,10.00,CLR,M,M,M,0.00,30.07
